# CVE vs CWE

## CVEs Download

In [ ]:
!git clone https://github.com/CVEProject/cvelistV5.git

## Import and install major libraries

In [ ]:
!pip install pandas numpy

In [ ]:
import pandas as pd
import xml.etree.ElementTree as ET
import json
import glob
import re

## Create Data Structures

In [ ]:
def parse_cwe_metadata(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    # Define namespaces if necessary, usually CWE XMLs use a specific one
    ns = {'cwe': 'http://cwe.mitre.org/cwe-7'} 
    
    cwe_data = []
    # In CWE XML, weaknesses are usually under a <Weaknesses> tag
    for weakness in root.findall('.//cwe:Weakness', ns):
        cwe_id = weakness.get('ID')
        cwe_data.append({
            'cwe_id': f"CWE-{cwe_id}",
            'name': weakness.get('Name'),
            'abstraction': weakness.get('Abstraction'),
            # You can add logic here to calculate depth/parents later
        })
    return pd.DataFrame(cwe_data)

df_cwe_metadata = parse_cwe_metadata('cwec_v4.19.1.xml')

In [ ]:
def extract_cwes(container):
    found =[]
    descriptions = [d for pt in container.get('problemTypes', []) for d in pt.get('descriptions',[])]
    
    for desc in descriptions:
        if "cweId" in desc:
            found.append(desc["cweId"])
            continue
            
        txt = desc.get('description', '')
        
        # 1. Match numeric IDs: handles spaces, colons, "ID", Unicode dashes (\u2011-\u2014), and URLs
        if match := re.search(r'(?:CWE[\s\-\u2011\u2013\u2014:]*(?:ID\s*)?|definitions/)(\d+)', txt, re.IGNORECASE):
            found.append(f"CWE-{match.group(1)}")
            continue
        # 2. Fallback: If no digits are found but it mentions CWE, Unknown, Other, etc., map to noinfo
        # \b ensures we match whole words (so we don't accidentally match "CWE" inside "macweep")
        elif re.search(r'(?i)\b(?:NOINFO|OTHER|UNKNOWN|CWE|NO_CWE)\b', txt):
            found.append("CWE-noinfo")
            continue
        elif "CWE" in txt.upper():
            print(desc)
            
    return list(set(found))

def parse_all_cves(base_path):
    records = []
    # Use glob to find all json files in the nested structure
    file_paths = glob.glob(f"{base_path}/**/*.json", recursive=True)
    
    for path in file_paths:
        with open(path, 'r', encoding='utf-8') as f:
            try:
                data = json.load(f)
                cve_id = data['cveMetadata']['cveId']
                published_date = data['cveMetadata'].get('datePublished', None)

                # Extract from CNA
                cna_cwes = extract_cwes(data['containers'].get('cna', {}))
                # Extract from ADP (there might be multiple ADPs)
                adp_cwes = []
                for adp in data['containers'].get('adp', []):
                    adp_cwes.extend(extract_cwes(adp))

                # Store result
                records.append({
                    'cve_id': cve_id,
                    'year': cve_id.split('-')[1],
                    'published_date': published_date,
                    'cna_cwes': cna_cwes,
                    'adp_cwes': adp_cwes,
                    'has_cwe': len(cna_cwes) > 0 or len(adp_cwes) > 0
                })
            except Exception as e:
                continue # Skip malformed files
                
    return pd.DataFrame(records)

df_cve = parse_all_cves('cvelistV5/cves')

In [18]:
len(df_cwe_metadata)

969

In [19]:
len(df_cve)

341110

## Metrics calculation

In [ ]:
#df_cve, df_cwe_metadata